# Generate event-aligned h5 datasets

New-pipeline version of `OLDPIPELINEFILES/generate_h5_dataset.ipynb`: load a session, choose units by their quality metrics, align spikes to task events, and save one h5 file per event.

To process many sessions without the notebook, list them in a config file and run `python scripts/generate_h5_dataset.py config.toml` (see `scripts/example_config.toml`).

In [1]:
from pathlib import Path

import numpy as np

from ephys_behavior_formatting import AlignedSpikes, NeuropixelsExperiment, PlexonExperiment, UnitCriteria

/home/mateoumaguing/anaconda3/envs/ephys-fmt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Neuropixels sessions

## Load the session

`kilosort_dir` is the Kilosort output folder. The raw `.ap.bin`/`.ap.meta` files are found there or in its parent folder (pass `raw_dir=` otherwise).

In [2]:
monkey_id = "B"
session_date = "20250325"

ssd = Path("/media/mateoumaguing/Extreme SSD")
kilosort_dir = ssd / "example_npix_session/BILBO_CHKDLAY_DLPFC_03252025_g0/BILBO_CHKDLAY_DLPFC_03252025_g0_imec0/kilosort4"
events_path = ssd / f"CFD_DLPFC/BilboEvents/{session_date}/CFDeventStruct.mat"

experiment = NeuropixelsExperiment(
    kilosort_dir,
    events_path,
    monkey_id=monkey_id,
    session_date=session_date,
    roi="DLPFC",
    task_type="CHKDLAY",
)

## Load units

Reads every unit's spike times and a sample of its waveforms (used for SNR and amplitude cutoff). Restrict with `unit_ids=` or `quality_labels=("good",)`.

In [3]:
experiment.load_units(n_jobs=8);

Loading units: 100%|██████████| 392/392 [03:15<00:00,  2.01it/s]


Loaded 392 of 392 units.


## Trial parameters

In [4]:
trials = experiment.trial_params_frame()
print(f"{len(trials)} trials")
trials.head(10)

645 trials


,trial_outcomes,trial_RTs,trial_action_choices,trial_cues,trial_configs,trial_color_choices,trial_delay_lengths,trial_savetags,trial_unique_ids
0,incorrect,787,right,135,RG,green,601,1,63910262980
1,correct,769,right,45,RG,green,515,1,63910262988
2,correct,1042,left,90,GR,green,703,1,63910262993
3,correct,826,left,108,GR,green,567,1,63910262998
4,correct,775,right,180,GR,red,483,1,63910263002
5,correct,796,left,11,GR,green,593,1,63910263007
6,incorrect,847,left,147,GR,green,607,1,63910263011
7,incorrect,824,right,78,GR,red,539,1,63910263020
8,correct,850,right,67,RG,green,790,1,63910263028
9,correct,708,left,135,RG,red,483,1,63910263033


## Choose units by quality metrics

Every criterion left unset is ignored. `include_ids` / `exclude_ids` override the criteria for hand-picked units.

In [5]:
metrics = experiment.unit_metrics()
metrics.describe()

/home/mateoumaguing/Documents/MiscData/ephys-behavior-formatting/src/ephys_behavior_formatting/units.py:156: UserWarning: Unit 376: amplitude pdf has no unique minimum; amplitude_cutoff may be unreliable with this few waveforms.
  "amplitude_cutoff": self.amplitude_cutoff(),


,num_spikes,firing_rate,isi_mean,isi_std,isi_violations_ratio,isi_violations_count,presence_ratio,amplitude_cutoff,snr
count,392.000000,392.000000,391.000000,391.000000,392.000000,392.000000,391.000000,392.000000,391.000000
mean,11132.362245,2.819038,7.304244,10.451450,1.533412,49.456633,0.971204,0.048049,2.047653
std,13661.857156,3.457475,51.165411,63.341956,12.956431,117.619931,0.114051,0.079518,1.302878
min,1.000000,0.000000,0.058730,0.079820,0.000000,0.000000,0.066667,0.000000,0.595182
25%,1279.500000,0.324541,0.223005,0.365324,0.000000,0.000000,1.000000,0.010795,1.317530
50%,4908.000000,1.242813,0.799354,1.356263,0.136697,2.500000,1.000000,0.022129,1.647051
75%,17666.500000,4.473522,3.073476,4.287258,0.347106,33.250000,1.000000,0.052900,2.244729
max,67278.000000,17.027290,913.490302,1058.868932,219.831895,878.000000,1.000000,0.500000,10.216527


In [6]:
criteria = UnitCriteria(
    quality_labels=("good", "mua"),
    min_firing_rate=1.0,          # Hz
    max_isi_violations_ratio=1,
    min_presence_ratio=0.9,
    # min_snr=3.0,
    # max_amplitude_cutoff=0.1,
    # include_ids=[...],
    # exclude_ids=[...],
)
selected = experiment.select_units(criteria)
print(f"{len(selected)} of {len(experiment.units)} units pass")

204 of 392 units pass


## Align spikes and save

Each file holds `spike_counts` with shape (trials, units, time bins), plus trial parameters, unit ids, unit metrics and the criteria used.

In [7]:
savepath = Path("/home/mateoumaguing/Documents/MiscData/CFD_data_h5")
overwrite = True  # replace files from earlier runs of this notebook

alignments = [
    {"event_name": "CheckerboardDrawnTime", "pre_time": 0.400, "post_time": 2.000},
    {"event_name": "TargetsDrawnTime", "pre_time": 1.800, "post_time": 1.000},
]

for alignment in alignments:
    aligned = experiment.align_spikes(**alignment, bin_size=0.001, criteria=criteria)
    path = aligned.save(savepath, overwrite=overwrite)
    print(f"{aligned.spike_counts.shape} -> {path}")

(645, 204, 2400) -> /home/mateoumaguing/Documents/MiscData/CFD_data_h5/B_CHKDLAY_DLPFC_20250325_CheckerboardAligned.h5
(645, 204, 2800) -> /home/mateoumaguing/Documents/MiscData/CFD_data_h5/B_CHKDLAY_DLPFC_20250325_TargetsAligned.h5


## Load a saved file

In [8]:
loaded = AlignedSpikes.load(savepath / f"{monkey_id}_CHKDLAY_DLPFC_{session_date}_CheckerboardAligned.h5")

print(loaded.metadata)
print(loaded.spike_counts.shape)  # (trials, units, time bins)
loaded.trial_params_frame().head()

{'monkey_id': 'B', 'session_date': '20250325', 'roi': 'DLPFC', 'task_type': 'CHKDLAY', 'probe_type': 'Neuropixels'}
(645, 204, 2400)


,trial_RTs,trial_action_choices,trial_color_choices,trial_configs,trial_cues,trial_delay_lengths,trial_outcomes,trial_savetags,trial_unique_ids
trial_index,,,,,,,,,
0,787,right,green,RG,135,601,incorrect,1,63910262980
1,769,right,green,RG,45,515,correct,1,63910262988
2,1042,left,green,GR,90,703,correct,1,63910262993
3,826,left,green,GR,108,567,correct,1,63910262998
4,775,right,red,GR,180,483,correct,1,63910263002


In [9]:
# e.g. correct trials only, from -400 to 900 ms
window = loaded.select_trials(loaded.trial_params["trial_outcomes"] == "correct").time_window(-0.400, 0.900)
window.spike_counts.shape

(504, 204, 1300)

# Plexon sessions

Same steps for a Plexon session. `PlexonExperiment` starts from the session's `allTrials` file (assembled from the raw per-trial `.dat` files by `PlexonSessionProcessor`, see the end of this notebook).

Plexon files have no hand-movement event, so reaction times come from the session's `CFDeventStruct.mat` (`events_path`), matched to the Plexon trials by unique id. Without it, `trial_RTs` is NaN.

## Load the session

In [ ]:
plexon_monkey_id = "T"
plexon_session_date = "20220309"

plexon = PlexonExperiment(
    ssd / f"CFD_plexon_data/TiberiusCHKDLAY{plexon_session_date}.mat",
    monkey_id=plexon_monkey_id,
    session_date=plexon_session_date,
    roi="DLPFC",
    task_type="CHKDLAY",
    events_path=ssd / f"CFD_DLPFC/TiberiusEvents/{plexon_session_date}/CFDeventStruct.mat",
    # valid_savetags=[1, 2],     # keep only these recording blocks
    # include_unsorted=True,     # also treat each channel's unsorted spikes (unit 0) as a unit
)

## Load units

Unit ids are `channel * 100 + unit` (e.g. 1302 = channel 13, unit 2).

In [ ]:
plexon.load_units();

## Trial parameters

In [ ]:
plexon_trials = plexon.trial_params_frame()
print(f"{len(plexon_trials)} trials, {plexon_trials['trial_RTs'].isna().sum()} without an RT")
plexon_trials.head(10)

## Choose units by quality metrics

Two metrics behave differently for Plexon data:
- **ISI violations** are not meaningful: spike times are xPC packet timestamps with ~1 ms resolution.
- **Amplitude cutoff** is usually at its 0.5 cap: Plexon only stores threshold-crossing snippets, so the low-amplitude side of the distribution is missing.

In [ ]:
plexon.unit_metrics().describe()

In [ ]:
plexon_criteria = UnitCriteria(
    min_firing_rate=1.0,  # Hz
    min_presence_ratio=0.9,
    # min_snr=2.0,
    # include_ids=[...],
    # exclude_ids=[...],
)
plexon_selected = plexon.select_units(plexon_criteria)
print(f"{len(plexon_selected)} of {len(plexon.units)} units pass")

## Align spikes and save

In [ ]:
for alignment in alignments:
    aligned = plexon.align_spikes(**alignment, bin_size=0.001, criteria=plexon_criteria)
    path = aligned.save(savepath, overwrite=overwrite)
    print(f"{aligned.spike_counts.shape} -> {path}")

## Load a saved file

In [ ]:
plexon_loaded = AlignedSpikes.load(
    savepath / f"{plexon_monkey_id}_CHKDLAY_DLPFC_{plexon_session_date}_TargetsAligned.h5"
)
print(plexon_loaded.metadata)
print(plexon_loaded.spike_counts.shape)  # (trials, units, time bins)
plexon_loaded.trial_params_frame().head()

## Building `allTrials` from raw `.dat` files

Only needed for sessions that haven't been processed yet. Requires MATLAB and the MATLAB Engine for Python (`scripts/install_matlab_engine.sh`).

In [ ]:
from ephys_behavior_formatting.plexon import PlexonSessionProcessor

# processor = PlexonSessionProcessor("/path/to/raw/20220309", task_type="CHKDLAY")
# processor.process(include_spikes=True, include_lfp=False)
# processor.save(ssd / "CFD_plexon_data", monkey_id="T", date="20220309")